# Local Deployment

#### <i><b>NOTE: For the local deployment part, chatGPT was used for text and code generation</b></i>

1. Typical air quality use cases:   
    * Anomaly detection (e.g. pollution spikes)
	* Station segmentation
	* Temporal pattern discovery (rush hour vs night)
	* Early warning systems

    👉 Key idea:
   * Batch use → dashboards
   * Real-time use → API / streaming
  
2. Standard deployment architectures

    <b>Option A — Batch (simple & very common)</b>

    Best for reporting and monitoring

    Pipeline:
        1.	New data arrives (e.g., daily)
        2.	Feature engineering
        3.	Apply clustering
        4.	Store results
        5.	Visualize in dashboard

    Typical tools: Scheduler: cron / Airflow, Storage: PostgreSQL / BigQuery, Dashboard: Tableau or Power BI    
        👉 Pros: robust, simple    
        👉 Cons: not real-time   

    <b>Option B — API (for real-time use cases)</b>

    Live classification of incoming sensor data

    How it works: Model is served as a service, Input: new sensor data, Output: cluster ID + interpretation

    Typical stack: Backend: FastAPI or Flask, Packaging: Docker, Cloud: AWS or Google Cloud     
        👉 Important: clustering models are not stable like supervised models → versioning is critical      


#### How to test API?
* Simply run all code blocks in this notebook from top to bottom
* Open http://localhost:8000/docs#/default/predict_predict_post for testing the API
* Test JSON: {"data": [2.7, 15.1, 122.0, 1096.0, 7, 8, 27.0, 35.3]}
* Receive a color indicating the air quality of the submitted data

In [1]:
import pandas as pd
import pickle

# Generate Test Data for Swagger API
df: pd.DataFrame = pd.read_csv('../data/AirQualityUCIClustered.csv')
df["AirQuality"] = df["Cluster"].map({0: "yellow", 1: "red", 2: "green"})

# Pick 5 rows of each AirQuality color
df_5_each = df.groupby('AirQuality').sample(n=5, random_state=42)

# Filter columns by using only features that were used for training
features = pickle.load(open('../data/features.pkl', 'rb'))
df_test_swagger: pd.DataFrame = df_5_each.filter(items=features + ['AirQuality'])

# Test dict
test_dict: dict[list: str] = {}

# Print list for testing API
for row in df_test_swagger.values.tolist():
    test_data_cluster = row[-1]         # e.g. 'green'
    test_data_row = row[:len(row)-1]    # e.g. [2.7, 15.1, 122.0, 1096.0, 7, 8, 27.0, 35.3]
    if test_data_cluster not in test_dict:
        test_dict[test_data_cluster] = test_data_row # e.g. [2.7, 15.1, 122.0, 1096.0, 7, 8, 27.0, 35.3] -> green
    print(f'{test_data_row} -> {test_data_cluster}')

# Test dict
test_dict

[2.7, 15.1, 122.0, 1096.0, 7, 8, 27.0, 35.3] -> green
[3.0, 16.3, 136.0, 1125.0, 9, 13, 30.2, 30.2] -> green
[1.8, 6.1, 134.0, 1111.0, 11, 23, 10.2, 49.4] -> green
[2.08, 19.5, 86.57, 1101.0, 6, 19, 25.3, 30.0] -> green
[2.8, 9.7, 150.0, 1062.0, 3, 18, 22.7, 42.4] -> green
[0.6, 1.4, 50.0, 375.0, 4, 7, 13.5, 48.6] -> red
[1.9, 9.8, 62.0, 1441.0, 11, 9, 20.3, 71.1] -> red
[0.7, 1.5, 79.0, 399.0, 3, 9, 7.3, 48.7] -> red
[0.7, 2.6, 33.0, 516.0, 8, 2, 26.7, 61.0] -> red
[0.9, 2.0, 85.0, 377.0, 3, 11, 11.2, 33.8] -> red
[5.9, 30.8, 200.0, 1947.0, 3, 19, 20.1, 43.2] -> yellow
[4.1, 20.81, 170.0, 1622.72, 12, 13, 14.73, 60.01] -> yellow
[4.7, 26.7, 177.0, 1600.0, 9, 20, 27.1, 42.2] -> yellow
[2.16, 30.7, 85.5, 1562.0, 8, 8, 23.8, 56.5] -> yellow
[3.17, 18.6, 108.95, 1360.0, 9, 15, 24.6, 63.9] -> yellow


{'green': [2.7, 15.1, 122.0, 1096.0, 7, 8, 27.0, 35.3],
 'red': [0.6, 1.4, 50.0, 375.0, 4, 7, 13.5, 48.6],
 'yellow': [5.9, 30.8, 200.0, 1947.0, 3, 19, 20.1, 43.2]}

In [2]:
import joblib
import numpy as np
from sklearn.preprocessing import StandardScaler


def get_color(cluster: int):
    if cluster == 0:
        return 'yellow'
    elif cluster == 1:
        return 'red'
    elif cluster == 2:
        return 'green'
    else:
        return 'Unknown Cluster'


model = joblib.load("../models/kmeans.pkl")
scaler: StandardScaler = joblib.load("../models/scaler.pkl")

for cluster_color, data_row in test_dict.items():
    # print(data_row, cluster_color)

    # Testing
    X = pd.DataFrame([data_row], columns=features)
    X_scaled = pd.DataFrame(scaler.transform(X))

    cluster = model.predict(X_scaled)
    cluster_number: int = int(cluster[0]) # array([<cluster-number>], dtype=int32)
    if get_color(cluster_number) == cluster_color:
        print(f'Prediction ({cluster_number}) was correct for data row: {data_row}')
    else:
        print(get_color(cluster_number))

Prediction (2) was correct for data row: [2.7, 15.1, 122.0, 1096.0, 7, 8, 27.0, 35.3]
Prediction (1) was correct for data row: [0.6, 1.4, 50.0, 375.0, 4, 7, 13.5, 48.6]
Prediction (0) was correct for data row: [5.9, 30.8, 200.0, 1947.0, 3, 19, 20.1, 43.2]


### Option B - API

In [3]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI
import joblib
import numpy as np
import pandas as pd
import pickle
from pydantic import BaseModel
from typing import List

class InputData(BaseModel):
    data_row: List[float]

nest_asyncio.apply()

app = FastAPI()
model = joblib.load("../models/kmeans.pkl")
scaler = joblib.load("../models/scaler.pkl")
features: list = pickle.load(open('../data/features.pkl', 'rb'))


@app.post("/predict")
def predict(input: InputData):

    # Convert to DataFrame with correct feature names
    X = pd.DataFrame([input.data_row], columns=features)

    # Ensure numeric (handles booleans too)
    X = X.astype(float)

    # Scale (DO NOT refit)
    X_scaled = scaler.transform(X)

    # Predict
    cluster = model.predict(X_scaled)

    return {"cluster": get_color(int(cluster[0]))}

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

await server.serve()

# http://localhost:8000/docs#/default/predict_predict_post for testing API

INFO:     Started server process [7583]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:49987 - "GET / HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:49989 - "GET /apple-touch-icon-precomposed.png HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:49987 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:49989 - "GET /apple-touch-icon.png HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:49989 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:49992 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:49992 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:49994 - "GET /apple-touch-icon-precomposed.png HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:49995 - "GET /apple-touch-icon.png HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:49996 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:50013 - "POST /predict HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [7583]
